# 06 — Lightweight Backtest Simulation

⚠️ **READ BEFORE INTERPRETING RESULTS** ⚠️

This is a **pre-cost gross simulation only**. It does NOT model:
- Bid-ask spread or market impact
- Rebalancing transaction costs (configurable placeholder only)
- Short-selling costs or constraints
- Liquidity limits or position sizing
- Survivorship-free index membership
- Actual execution prices (assumes perfect fill at close)

**Purpose:** Verify the signal produces directionally sensible portfolio behaviour. Results are NOT evidence of a viable trading strategy.

**The correct takeaway:** If the portfolio beats SPY on a gross pre-cost basis, the signal has directionality. Estimating post-cost performance requires institutional-grade infrastructure not available in this project.

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as cfg
from src.backtest import run_backtest, turnover_adjusted_spread
from src.plotting import plot_cumulative_returns

RAW       = cfg.PROJECT_ROOT / cfg.get('data.raw_dir')
PROCESSED = cfg.processed_dir()
print('OK')

In [ ]:
# Load OOS scores saved from notebook 05
# If you haven't saved them, re-run the walk-forward and save:
# oos_scores.to_parquet(PROCESSED / 'oos_scores.parquet')

try:
    oos_scores = pd.read_parquet(PROCESSED / 'oos_scores.parquet')
    print(f'OOS scores loaded: {len(oos_scores):,} rows')
except FileNotFoundError:
    print('Run notebook 05 first and save: oos_scores.to_parquet(PROCESSED / "oos_scores.parquet")')

In [ ]:
# Load raw prices for return computation
prices    = pd.read_parquet(RAW / 'prices.parquet')
benchmark = pd.read_parquet(RAW / 'benchmark.parquet')
print(f'Prices: {prices.shape}')
print(f'Benchmark: {benchmark.shape}')

In [ ]:
# Run backtest — zero cost baseline
bt = run_backtest(
    scores               = oos_scores,
    prices               = prices,
    benchmark            = benchmark,
    top_pct              = cfg.get('backtest.top_quantile', 0.20),
    rebalance_freq       = 'ME',
    transaction_cost_bps = 0,  # gross — no costs
)

print('\n=== BACKTEST SUMMARY (PRE-COST) ===')
for k, v in bt['stats'].items():
    if k != 'disclaimer':
        print(f'  {k:<30} {v}')
print(f'\n⚠️  {bt["stats"]["disclaimer"]}')

In [ ]:
# Plot cumulative returns
fig = plot_cumulative_returns(bt['cumulative'], bt['stats'])
plt.show()

## Transaction Cost Sensitivity

In [ ]:
# Show how performance degrades with increasing transaction costs
costs_bps = [0, 5, 10, 20, 30]
rows = []

for bps in costs_bps:
    adj_returns = turnover_adjusted_spread(
        bt['portfolio_returns'],
        bt['turnover'],
        cost_bps=bps,
    )
    cum = (1 + adj_returns).cumprod()
    total_ret = (cum.iloc[-1] - 1) * 100 if not cum.empty else np.nan
    rows.append({'cost_bps': bps, 'total_return_pct': round(total_ret, 2)})

tc_sensitivity = pd.DataFrame(rows)
print('Transaction cost sensitivity:')
print(tc_sensitivity.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tc_sensitivity['cost_bps'], tc_sensitivity['total_return_pct'],
        marker='o', color='#185FA5', linewidth=2)
ax.axhline(0, color='#A32D2D', linestyle='--', linewidth=1)
ax.set_xlabel('One-way transaction cost (bps)')
ax.set_ylabel('Total return (%)')
ax.set_title('Return vs Transaction Cost Assumption\n(Approximate — assumes constant turnover)')
plt.tight_layout()
plt.show()

## Interpretation Guide

1. **If pre-cost total return >> SPY**: Signal has directional quality. Post-cost depends on turnover and cost assumptions — both highly uncertain without production infrastructure.

2. **If pre-cost total return ≈ SPY**: Signal adds limited value in portfolio form. Check if IC is positive — IC can be positive even when portfolio returns don't outperform (due to diversification effects).

3. **If pre-cost total return << SPY**: Signal quality is insufficient. Review feature IC — if IC is also low, feature re-examination needed.

**Bottom line:** The IC and ICIR metrics from notebook 05 are the primary quality signals. The backtest here is illustrative only.